In [ ]:
import glob
import json
import math
import re
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np
from scipy.io import loadmat
from scipy.optimize import linear_sum_assignment

NEW_FILE_PATTERNS = [
    '/Users/savirmadan/Partners HealthCare Dropbox/Savir Madan/SEEGAnalysis/NewLeGui/*Output/derivatives/leaddbs/sub-*/reconstruction/sub-*_electrodes.mat',
    '/Users/savirmadan/Partners HealthCare Dropbox/Savir Madan/SEEGAnalysis/NewLeGui/*Ouput/derivatives/leaddbs/sub-*/reconstruction/sub-*_electrodes.mat',
]
OLD_FILE_PATTERNS = [
    '/Users/savirmadan/Partners HealthCare Dropbox/Savir Madan/SEEGAnalysis/OLD_LEGUI_TESTING_FOLDER/sub*/Registered/Electrodes.mat',
    '/Users/savirmadan/Partners HealthCare Dropbox/Savir Madan/SEEGAnalysis/OLD_LEGUI_TESTING_FOLDER/sub-*/Registered/Electrodes.mat',
]

# Projected variants are exact duplicates of the raw variants in these files, so we test one representative per family.
VARIABLE_FAMILIES = [
    {
        'variable': 'ElecXYZRaw',
        'family_label': 'ElecXYZRaw',
        'duplicate_variables': ['ElecXYZProjRaw'],
        'unit_kind': 'mm',
        'allow_ct_backwarp': True,
    },
    {
        'variable': 'ElecXYZMNIRaw',
        'family_label': 'ElecXYZMNIRaw',
        'duplicate_variables': ['ElecXYZMNIProjRaw'],
        'unit_kind': 'mm',
        'allow_ct_backwarp': False,
    },
    {
        'variable': 'ElecCOMIdxRaw',
        'family_label': 'ElecCOMIdxRaw',
        'duplicate_variables': ['ElecCOMIdxProjRaw'],
        'unit_kind': 'voxel_index',
        'allow_ct_backwarp': False,
    },
]

EXCLUDED_VARIABLE_NOTES = {
    'ElecMapRaw': 'Old files do not carry usable unique contact labels here; many entries are NaN or malformed.',
    'ElecTypeRaw / ElecTypeProjRaw': 'Only tissue classes such as Gray/White/Unknown, not unique contact identities.',
    'ElecAtlasRaw / ElecAtlasProjRaw / ElecAtlasProbProjRaw': 'Atlas annotations are anatomical summaries, not one-to-one contact identifiers.',
    'ElecFullIdxRaw / ElecFullIdxProjRaw': 'Per-contact voxel clouds; their centroids are already summarized by ElecCOMIdxRaw.',
    'DepthElecRaw / GndElecRaw / RefElecRaw / MicroElecRaw': 'Metadata rather than geometric contact coordinates.',
}


In [ ]:
def subject_id_from_path(path):
    match = re.search(r'sub-sub(\d+)|sub-(\d+)|sub(\d+)', path)
    if not match:
        raise ValueError(f'Could not parse subject id from: {path}')
    return next(group for group in match.groups() if group is not None)


def choose_largest_file_per_subject(paths):
    chosen_paths = {}
    for path in paths:
        subject = subject_id_from_path(path)
        if subject not in chosen_paths or Path(path).stat().st_size > Path(chosen_paths[subject]).stat().st_size:
            chosen_paths[subject] = path
    return chosen_paths


def find_subject_file_pairs():
    new_files = sorted({path for pattern in NEW_FILE_PATTERNS for path in glob.glob(pattern)})
    old_files = sorted({path for pattern in OLD_FILE_PATTERNS for path in glob.glob(pattern)})

    new_map = choose_largest_file_per_subject(new_files)
    old_map = choose_largest_file_per_subject(old_files)
    overlap_subjects = sorted(set(old_map) & set(new_map))

    return overlap_subjects, old_map, new_map


def load_vector_dataset(path, variable_name):
    with h5py.File(path, 'r') as handle:
        if variable_name not in handle:
            return None
        values = np.asarray(handle[variable_name][()], dtype=float)

    if values.ndim != 2:
        return None
    if values.shape[0] == 3:
        values = values.T
    elif values.shape[1] != 3:
        return None

    return values


def max_abs_difference(path, left_variable, right_variable):
    left = load_vector_dataset(path, left_variable)
    right = load_vector_dataset(path, right_variable)
    if left is None or right is None:
        return math.nan
    return float(np.max(np.abs(left - right)))


def list_ct_backwarp_transforms(new_electrode_path, subject):
    subject_root = Path(new_electrode_path).parents[1]
    transform_dir = subject_root / 'coregistration' / 'transformations'
    return sorted(transform_dir.glob(f'sub-sub{subject}_from-anchorNative_to-CT_desc-*44.mat'))


def transform_name_from_path(path):
    match = re.search(r'desc-(.*)44\.mat$', Path(path).name)
    if not match:
        raise ValueError(f'Could not parse transform name from: {path}')
    return match.group(1)


def load_affine44(path):
    return np.asarray(loadmat(path, squeeze_me=True)['tmat'], dtype=float)


def apply_affine_to_points(points, affine):
    homogeneous_points = np.column_stack([points, np.ones(len(points))])
    transformed_points = homogeneous_points @ affine.T
    return transformed_points[:, :3]


def assignment_metrics(old_points, new_points):
    pairwise_distances = np.linalg.norm(old_points[:, None, :] - new_points[None, :, :], axis=2)
    assigned_old_idx, assigned_new_idx = linear_sum_assignment(pairwise_distances)
    assigned_distances = pairwise_distances[assigned_old_idx, assigned_new_idx]

    assignment_matches = [
        {
            'old_contact_idx': int(old_idx),
            'new_contact_idx': int(new_idx),
            'distance': float(distance),
        }
        for old_idx, new_idx, distance in zip(assigned_old_idx, assigned_new_idx, assigned_distances)
    ]
    assignment_matches.sort(key=lambda row: row['distance'])

    return {
        'assigned_pairs': int(len(assignment_matches)),
        'old_unmatched': int(len(old_points) - len(assignment_matches)),
        'new_unmatched': int(len(new_points) - len(assignment_matches)),
        'assignment_total': float(assigned_distances.sum()),
        'assignment_mean': float(assigned_distances.mean()),
        'assignment_median': float(np.median(assigned_distances)),
        'assignment_max': float(assigned_distances.max()),
        'assignment_within_2': float((assigned_distances <= 2.0).mean()),
        'assignment_within_5': float((assigned_distances <= 5.0).mean()),
        'assignment_matches': assignment_matches,
    }


def candidate_sort_key(result):
    return (result['assignment_mean'], result['assignment_median'], result['assignment_max'])


def read_pipeline_ct_method(new_electrode_path, subject):
    subject_root = Path(new_electrode_path).parents[1]
    log_path = subject_root / 'coregistration' / 'log' / f'sub-sub{subject}_desc-coregmethod.json'
    log_data = json.loads(log_path.read_text())
    return log_data['method']['CT']


def evaluate_candidate(subject, old_path, new_path, family_spec, transform_name='direct', transform_path=''):
    old_points = load_vector_dataset(old_path, family_spec['variable'])
    new_points = load_vector_dataset(new_path, family_spec['variable'])
    if old_points is None or new_points is None:
        return None

    if transform_path:
        affine = load_affine44(transform_path)
        new_points = apply_affine_to_points(new_points, affine)

    result = assignment_metrics(old_points, new_points)
    result.update(
        {
            'subject': subject,
            'variable': family_spec['variable'],
            'family_label': family_spec['family_label'],
            'duplicate_variables': family_spec['duplicate_variables'],
            'unit_kind': family_spec['unit_kind'],
            'transform_name': transform_name,
            'transform_path': transform_path,
            'candidate_label': f"{family_spec['family_label']}|{transform_name}",
        }
    )
    return result


def analyze_subject(subject, old_path, new_path):
    results = []
    for family_spec in VARIABLE_FAMILIES:
        direct_result = evaluate_candidate(subject, old_path, new_path, family_spec, transform_name='direct')
        if direct_result is not None:
            results.append(direct_result)

        if family_spec['allow_ct_backwarp']:
            for transform_path in list_ct_backwarp_transforms(new_path, subject):
                results.append(
                    evaluate_candidate(
                        subject,
                        old_path,
                        new_path,
                        family_spec,
                        transform_name=transform_name_from_path(transform_path),
                        transform_path=str(transform_path),
                    )
                )

    mm_results = [result for result in results if result['unit_kind'] == 'mm']
    index_results = [result for result in results if result['unit_kind'] == 'voxel_index']

    return {
        'subject': subject,
        'old_path': old_path,
        'new_path': new_path,
        'pipeline_ct_method': read_pipeline_ct_method(new_path, subject),
        'all_results': sorted(results, key=candidate_sort_key),
        'mm_results': sorted(mm_results, key=candidate_sort_key),
        'index_results': sorted(index_results, key=candidate_sort_key),
        'best_mm_result': min(mm_results, key=candidate_sort_key) if mm_results else None,
        'best_index_result': min(index_results, key=candidate_sort_key) if index_results else None,
    }


In [ ]:
overlap_subjects, old_file_map, new_file_map = find_subject_file_pairs()
print('Overlapping subjects:', overlap_subjects)

print('\nDuplicate-variable check (max absolute difference across paired representations):')
for family_spec in VARIABLE_FAMILIES:
    for duplicate_variable in family_spec['duplicate_variables']:
        max_difference = 0.0
        for subject in overlap_subjects:
            max_difference = max(max_difference, max_abs_difference(old_file_map[subject], family_spec['variable'], duplicate_variable))
            max_difference = max(max_difference, max_abs_difference(new_file_map[subject], family_spec['variable'], duplicate_variable))
        print(f"  {family_spec['variable']} vs {duplicate_variable}: max abs diff = {max_difference:.6f}")

print('\nVariables inspected but excluded from matching:')
for variable_name, note in EXCLUDED_VARIABLE_NOTES.items():
    print(f'  {variable_name}: {note}')

results = {}
for subject in overlap_subjects:
    results[subject] = analyze_subject(subject, old_file_map[subject], new_file_map[subject])


def format_value(value):
    if isinstance(value, float):
        return f'{value:.3f}'
    return str(value)


def print_summary_table(rows):
    headers = list(rows[0].keys())
    widths = {header: max(len(header), max(len(format_value(row[header])) for row in rows)) for header in headers}
    print('  '.join(header.ljust(widths[header]) for header in headers))
    print('  '.join('-' * widths[header] for header in headers))
    for row in rows:
        print('  '.join(format_value(row[header]).ljust(widths[header]) for header in headers))


subject_summary_rows = []
for subject in overlap_subjects:
    subject_result = results[subject]
    best_mm_result = subject_result['best_mm_result']
    raw_xyz_result = next(result for result in subject_result['mm_results'] if result['candidate_label'] == 'ElecXYZRaw|direct')
    mni_xyz_result = next(result for result in subject_result['mm_results'] if result['candidate_label'] == 'ElecXYZMNIRaw|direct')

    subject_summary_rows.append(
        {
            'subject': subject,
            'pipeline_ct': subject_result['pipeline_ct_method'],
            'best_mm': best_mm_result['candidate_label'],
            'best_mean': best_mm_result['assignment_mean'],
            'raw_xyz_mean': raw_xyz_result['assignment_mean'],
            'mni_xyz_mean': mni_xyz_result['assignment_mean'],
            'best_le5': best_mm_result['assignment_within_5'],
            'pairs': best_mm_result['assigned_pairs'],
            'old_unmatched': best_mm_result['old_unmatched'],
            'new_unmatched': best_mm_result['new_unmatched'],
        }
    )

print('\nBest mm-space candidate per subject:')
print_summary_table(subject_summary_rows)

candidate_aggregate = defaultdict(lambda: {'count': 0, 'sum_mean': 0.0, 'sum_median': 0.0, 'sum_le5': 0.0, 'wins': 0})
for subject in overlap_subjects:
    subject_result = results[subject]
    best_label = subject_result['best_mm_result']['candidate_label']
    for result in subject_result['mm_results']:
        aggregate = candidate_aggregate[result['candidate_label']]
        aggregate['count'] += 1
        aggregate['sum_mean'] += result['assignment_mean']
        aggregate['sum_median'] += result['assignment_median']
        aggregate['sum_le5'] += result['assignment_within_5']
        if result['candidate_label'] == best_label:
            aggregate['wins'] += 1

aggregate_rows = []
for candidate_label, aggregate in candidate_aggregate.items():
    aggregate_rows.append(
        {
            'candidate': candidate_label,
            'avg_mean': aggregate['sum_mean'] / aggregate['count'],
            'avg_median': aggregate['sum_median'] / aggregate['count'],
            'avg_le5': aggregate['sum_le5'] / aggregate['count'],
            'wins': aggregate['wins'],
        }
    )
aggregate_rows.sort(key=lambda row: (row['avg_mean'], row['avg_median']))

print('\nAggregate leaderboard across mm-space candidates:')
print_summary_table(aggregate_rows)

for subject in overlap_subjects:
    subject_result = results[subject]
    print(f'\nsub{subject} | pipeline CT coreg: {subject_result["pipeline_ct_method"]}')
    print('  Top mm-space candidates:')
    for result in subject_result['mm_results'][:5]:
        print(
            f"    {result['candidate_label']}: mean={result['assignment_mean']:.3f}, median={result['assignment_median']:.3f}, "
            f"<=5mm={result['assignment_within_5']:.3f}, pairs={result['assigned_pairs']}, "
            f"old_unmatched={result['old_unmatched']}, new_unmatched={result['new_unmatched']}"
        )
    if subject_result['index_results']:
        best_index = subject_result['best_index_result']
        print(
            f"  Best index-space candidate (not mm-comparable): {best_index['candidate_label']} -> "
            f"mean={best_index['assignment_mean']:.3f}, median={best_index['assignment_median']:.3f}"
        )

best_overall_subject = min(overlap_subjects, key=lambda subject: candidate_sort_key(results[subject]['best_mm_result']))
inspection_result = results[best_overall_subject]['best_mm_result']

print(f'\nBest overall mm-space agreement: sub{best_overall_subject} using {inspection_result["candidate_label"]}')
print('Old file:', results[best_overall_subject]['old_path'])
print('New file:', results[best_overall_subject]['new_path'])
if inspection_result['transform_path']:
    print('Transform:', inspection_result['transform_path'])
print('Closest assigned contact pairs (sorted by distance):')
for row in inspection_result['assignment_matches'][:20]:
    print(f"  old[{row['old_contact_idx']:>3}] <-> new[{row['new_contact_idx']:>3}] : {row['distance']:.3f}")

close_match_threshold = 5.0
close_pairs = [row for row in inspection_result['assignment_matches'] if row['distance'] <= close_match_threshold]
print(f'\nAssigned pairs within {close_match_threshold:.1f} units for sub{best_overall_subject}: {len(close_pairs)} / {inspection_result["assigned_pairs"]}')
for row in close_pairs[:20]:
    print(f"  old[{row['old_contact_idx']:>3}] <-> new[{row['new_contact_idx']:>3}] : {row['distance']:.3f}")
